# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row = one pseudonymized content item.

**Time Window:** A trailing 90-day window of search performance. 

For the starter dataset, this is a pre-aggregated snapshot. For the warehouse release, this will be implemented as a 90-day aggregation from `fact_content_daily_performance`, where the "decision point" is the end of the window, and we evaluate the CTR observed within that period.

In [2]:
import os
import pandas as pd

# Self-contained path resolution: works no matter what cell ran before,
# and no matter what folder Jupyter happens to launch from.
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
csv_path = os.path.join(project_root, "data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)

# Verify grain: one row per content_id
is_unique = df['content_id'].is_unique
print(f"Is content_id unique? {is_unique}")
print(f"Total content items: {len(df)}")
df[['content_id', 'client_id', 'impressions_90d']].head()

Is content_id unique? True
Total content items: 30000


,content_id,client_id,impressions_90d
0,content_304f48230142,client_f369cb89fc,3803
1,content_a1fb4e703a9e,client_4e07408562,15320
2,content_9aa793d4d895,client_7f2253d7e2,12581
3,content_331d6c4de07b,client_19581e27de,11751
4,content_d99b7a2d90ca,client_3fdba35f04,19140


## 2. Fields: feature / label / context / excluded

I have sorted the fields based on their role in predicting the CTR Opportunity Gap.

**Features (Input signals):**
- `impressions_90d`: Volume signal (essential for filtering noise).
- `avg_position`: Primary driver of expected CTR.
- `position_tier`: Categorical version of position for easier baseline modeling.
- `content_type`: Baseline engagement varies by type (e.g., blog vs. landing page).
- `main_intent`: User intent (informational vs transactional) affects click behavior.
- `word_count`: Proxy for content depth/value.
- `content_age_days`: Freshness signal.
- `engagement_rate`, `scroll_rate`: On-page behavior signals.

**Label / Proxy (Target):**
- `ctr`: We target the "Opportunity Gap" (Observed CTR vs Expected CTR). For the proxy, we focus on `ctr < 0.5%` given `impressions_90d >= 500`.

**Context (Grouping/Metadata):**
- `content_id`: Primary key.
- `client_id`: Used for group-holdout validation to ensure cross-client generalization.

**Excluded (Why):**
- `trend_direction`, `trend_pct`: These measure visibility *movement* over time, which is the target for the "Refresh" lane, but for the "CTR" lane, we care about the *absolute gap* in engagement at the current visibility level. Including them would be a proxy for a different problem.

In [3]:
# Defining the buckets for the contract
buckets = {
    'features': ['impressions_90d', 'avg_position', 'position_tier', 'content_type', 'main_intent', 'word_count', 'content_age_days', 'engagement_rate', 'scroll_rate'],
    'label': ['ctr'],
    'context': ['content_id', 'client_id'],
    'excluded': ['trend_direction', 'trend_pct']
}

# Check if all fields exist in the dataset
all_fields = df.columns.tolist()
for bucket, fields in buckets.items():
    missing = [f for f in fields if f not in all_fields]
    print(f"{bucket}: {len(missing)} missing fields {missing}")

features: 0 missing fields []
label: 0 missing fields []
context: 0 missing fields []
excluded: 0 missing fields []


## 3. Verify it with queries (grain, counts, missing values, windows)

I will now verify the distributional health of the features and the validity of the proxy target.

In [4]:
# 1. Verify Grain and Volume
print(f"Total rows: {len(df)}")
print(f"Unique Content IDs: {df['content_id'].nunique()}")

# 2. Verify Target (CTR Opportunity)
visible_mask = (df['impressions_90d'] >= 500)
low_ctr_mask = (df['ctr'] < 0.5)
opportunity_mask = visible_mask & low_ctr_mask

print(f"Pages with high visibility (>=500 imp): {visible_mask.sum()}")
print(f"Pages with high visibility AND low CTR (<0.5%): {opportunity_mask.sum()}")
print(f"Opportunity Rate among visible pages: {opportunity_mask.sum()/visible_mask.sum():.2%}")

# 3. Check Missingness for key features
feature_cols = buckets['features']
missing_stats = df[feature_cols].isna().sum()
print("\nMissing values per feature:")
print(missing_stats[missing_stats > 0])

# 4. Check Position Distribution
print("\nPosition Distribution:")
print(df['position_tier'].value_counts(normalize=True).sort_index())

Total rows: 30000
Unique Content IDs: 30000
Pages with high visibility (>=500 imp): 16726
Pages with high visibility AND low CTR (<0.5%): 14245
Opportunity Rate among visible pages: 85.17%

Missing values per feature:
main_intent    2374
word_count     7699
scroll_rate     125
dtype: int64

Position Distribution:
position_tier
deep        0.043967
page_1      0.393800
page_3_5    0.241400
striking    0.243467
top_3       0.077367
Name: proportion, dtype: float64


## 4. Data limits

Every dataset has boundaries. For this CTR Opportunity analysis, the primary limits are:

1. **The Unbalanced Panel**: In the full warehouse release, different clients have different history depths. A 90-day window for one client might be the only data available, while another has 17 months. This means our "baseline" for CTR must be relative to the client's own history or very carefully normalized.
2. **The 'Zero-Click' Blindspot**: Our data measures clicks. However, some queries are "zero-click" (the answer is in the snippet). A low CTR on such a page isn't an "opportunity" for the editor; it's just how the query behaves.
3. **GSC-Only Periods**: Early history for some clients lacks GA4 engagement data (scroll/sessions). I must use the `ga4_data_available` flag to avoid treating "no data" as "zero engagement."
4. **Observation vs Causality**: This data is a snapshot. I can observe that "Low CTR is associated with X," but I can never prove "Changing X will increase CTR" without an experiment.

In [5]:
# Verification of GA4 availability (conceptually, as this is the starter CSV)
# In the warehouse, we would use: df[df['ga4_data_available'] == False]
# In the starter CSV, we check for NaN in engagement rates as a proxy for missing tracking
missing_engagement = df['engagement_rate'].isna().sum()
print(f"Rows with missing engagement_rate: {missing_engagement}")
print(f"Percentage missing: {missing_engagement/len(df):.2%}")

Rows with missing engagement_rate: 0
Percentage missing: 0.00%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.